In [ ]:
# Stage 10A — Pageable vs pinned-memory HtoD transfer

import time
import numpy as np
import pandas as pd
import torch

NUM_FRAMES = 300
WARMUP = 30

frames10 = bench_frames[:NUM_FRAMES + WARMUP]

H = frames10[0].shape[0]
W = frames10[0].shape[1]
C = frames10[0].shape[2]

print("=" * 80)
print("STAGE 10A — PINNED HOST MEMORY")
print("=" * 80)

print("Frames :", len(frames10))
print("Shape  :", (H, W, C))
print("Bytes/frame:",
      H * W * C,
      f"({H * W * C / (1024**2):.2f} MiB)")

In [ ]:
# Reusable pinned CPU buffer
pinned_cpu = torch.empty(
    (H, W, C),
    dtype=torch.uint8,
    pin_memory=True
)

# Reusable GPU buffer
gpu_buffer = torch.empty(
    (H, W, C),
    dtype=torch.uint8,
    device="cuda"
)

copy_stream = torch.cuda.Stream()

print("Pinned:", pinned_cpu.is_pinned())

In [ ]:
pageable_times = []
pinned_dma_times = []
pinned_stage_times = []
pinned_full_times = []

# ------------------------------------------------------------
# Warmup
# ------------------------------------------------------------

for f in frames10[:WARMUP]:

    cpu_t = torch.from_numpy(f)

    # Pageable
    gpu_buffer.copy_(cpu_t)

    # Pageable -> pinned staging
    pinned_cpu.copy_(cpu_t)

    # Async pinned -> device
    with torch.cuda.stream(copy_stream):
        gpu_buffer.copy_(pinned_cpu, non_blocking=True)

torch.cuda.synchronize()


# ------------------------------------------------------------
# Timed benchmark
# ------------------------------------------------------------

for f in frames10[WARMUP:WARMUP + NUM_FRAMES]:

    cpu_t = torch.from_numpy(f)

    # ========================================================
    # A. Current pageable CPU -> GPU
    # ========================================================

    torch.cuda.synchronize()

    t0 = time.perf_counter()

    gpu_buffer.copy_(cpu_t)

    torch.cuda.synchronize()

    t1 = time.perf_counter()

    pageable_times.append(
        (t1 - t0) * 1000.0
    )


    # ========================================================
    # B. CPU pageable -> pinned staging copy
    # ========================================================

    t0 = time.perf_counter()

    pinned_cpu.copy_(cpu_t)

    t1 = time.perf_counter()

    pinned_stage_times.append(
        (t1 - t0) * 1000.0
    )


    # ========================================================
    # C. Already-pinned -> GPU
    #
    # CUDA events measure actual GPU-side DMA time.
    # ========================================================

    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)

    with torch.cuda.stream(copy_stream):

        start.record(copy_stream)

        gpu_buffer.copy_(
            pinned_cpu,
            non_blocking=True
        )

        end.record(copy_stream)

    end.synchronize()

    pinned_dma_times.append(
        start.elapsed_time(end)
    )


    # ========================================================
    # D. Real staged path:
    #
    # normal OpenCV/numpy frame
    #      ↓
    # pinned host buffer
    #      ↓
    # async HtoD
    #
    # Includes staging cost.
    # ========================================================

    torch.cuda.synchronize()

    t0 = time.perf_counter()

    pinned_cpu.copy_(cpu_t)

    with torch.cuda.stream(copy_stream):
        gpu_buffer.copy_(
            pinned_cpu,
            non_blocking=True
        )

    copy_stream.synchronize()

    t1 = time.perf_counter()

    pinned_full_times.append(
        (t1 - t0) * 1000.0
    )

In [ ]:
def report(name, values):
    a = np.asarray(values)

    print(
        f"{name:30s}"
        f"median={np.median(a):8.4f} ms | "
        f"p95={np.percentile(a,95):8.4f} ms | "
        f"mean={np.mean(a):8.4f} ms"
    )


print("\n" + "=" * 90)
print("STAGE 10A RESULTS")
print("=" * 90)

report(
    "Pageable HtoD",
    pageable_times
)

report(
    "Pinned staging CPU copy",
    pinned_stage_times
)

report(
    "Pinned HtoD DMA only",
    pinned_dma_times
)

report(
    "Staging + pinned HtoD",
    pinned_full_times
)


page_med = np.median(pageable_times)
dma_med = np.median(pinned_dma_times)
full_med = np.median(pinned_full_times)

print("\n" + "-" * 90)

print(
    f"Pinned DMA speedup vs pageable : "
    f"{page_med / dma_med:.2f}x"
)

print(
    f"Real staged speedup             : "
    f"{page_med / full_med:.2f}x"
)

In [ ]:
df_stage10a = pd.DataFrame({
    "pageable_h2d_ms": pageable_times,
    "pinned_staging_ms": pinned_stage_times,
    "pinned_dma_ms": pinned_dma_times,
    "pinned_full_ms": pinned_full_times,
})

df_stage10a.to_csv(
    "/kaggle/working/stage10a_pinned_memory.csv",
    index=False
)

print(
    "\nSaved: "
    "/kaggle/working/stage10a_pinned_memory.csv"
)

In [ ]:
# Stage 10B — Overlap pinned HtoD transfer with YOLO inference

import time
import numpy as np
import pandas as pd
import torch

NUM_FRAMES = 300
WARMUP = 30

frames = bench_frames[:NUM_FRAMES + WARMUP]

H, W, C = frames[0].shape

# ------------------------------------------------------------
# Double buffers
# ------------------------------------------------------------

pinned = [
    torch.empty((H, W, C), dtype=torch.uint8, pin_memory=True),
    torch.empty((H, W, C), dtype=torch.uint8, pin_memory=True)
]

gpu_raw = [
    torch.empty((H, W, C), dtype=torch.uint8, device="cuda"),
    torch.empty((H, W, C), dtype=torch.uint8, device="cuda")
]

gpu_pre = [
    torch.empty((1, 3, 384, 640), dtype=torch.float32, device="cuda"),
    torch.empty((1, 3, 384, 640), dtype=torch.float32, device="cuda")
]

copy_stream = torch.cuda.Stream()
infer_stream = torch.cuda.Stream()

copy_done = [
    torch.cuda.Event(),
    torch.cuda.Event()
]

# ------------------------------------------------------------
# Helper: launch fused kernel on supplied stream
# ------------------------------------------------------------

def launch_fused(src_gpu, out_gpu, stream):
    lib.launch_fused_yolo_preprocess(
        ctypes.c_void_p(src_gpu.data_ptr()),
        ctypes.c_void_p(out_gpu.data_ptr()),
        W,
        H,
        ctypes.c_void_p(stream.cuda_stream)
    )


# ------------------------------------------------------------
# Warmup
# ------------------------------------------------------------

for i in range(WARMUP):

    b = i % 2
    cpu_t = torch.from_numpy(frames[i])

    pinned[b].copy_(cpu_t)

    with torch.cuda.stream(copy_stream):
        gpu_raw[b].copy_(pinned[b], non_blocking=True)
        copy_done[b].record(copy_stream)

    with torch.cuda.stream(infer_stream):
        infer_stream.wait_event(copy_done[b])

        launch_fused(
            gpu_raw[b],
            gpu_pre[b],
            infer_stream
        )

        with torch.no_grad():
            _ = model.model(gpu_pre[b])

torch.cuda.synchronize()

print("Warmup complete.")

In [ ]:
sequential_times = []

for f in frames[WARMUP:WARMUP + NUM_FRAMES]:

    cpu_t = torch.from_numpy(f)

    torch.cuda.synchronize()
    t0 = time.perf_counter()

    # CPU -> pinned
    pinned[0].copy_(cpu_t)

    # pinned -> GPU
    with torch.cuda.stream(copy_stream):
        gpu_raw[0].copy_(
            pinned[0],
            non_blocking=True
        )

    copy_stream.synchronize()

    # fused preprocess + inference
    with torch.cuda.stream(infer_stream):

        launch_fused(
            gpu_raw[0],
            gpu_pre[0],
            infer_stream
        )

        with torch.no_grad():
            _ = model.model(gpu_pre[0])

    infer_stream.synchronize()

    t1 = time.perf_counter()

    sequential_times.append(
        (t1 - t0) * 1000.0
    )

In [ ]:
# ------------------------------------------------------------
# Overlapped steady-state pipeline
# ------------------------------------------------------------

pipeline_start = torch.cuda.Event(enable_timing=True)
pipeline_end = torch.cuda.Event(enable_timing=True)

# CPU-side total wall time
torch.cuda.synchronize()
wall_start = time.perf_counter()

# ------------------------------------------------------------
# Prime first frame
# ------------------------------------------------------------

first = torch.from_numpy(
    frames[WARMUP]
)

pinned[0].copy_(first)

with torch.cuda.stream(copy_stream):
    gpu_raw[0].copy_(
        pinned[0],
        non_blocking=True
    )
    copy_done[0].record(copy_stream)

# ------------------------------------------------------------
# Main pipeline
# ------------------------------------------------------------

for i in range(NUM_FRAMES):

    current = i % 2

    # Inference for current frame
    with torch.cuda.stream(infer_stream):

        infer_stream.wait_event(
            copy_done[current]
        )

        launch_fused(
            gpu_raw[current],
            gpu_pre[current],
            infer_stream
        )

        with torch.no_grad():
            _ = model.model(
                gpu_pre[current]
            )

    # While current frame is being processed,
    # stage + copy next frame.
    if i + 1 < NUM_FRAMES:

        nxt = (i + 1) % 2

        cpu_next = torch.from_numpy(
            frames[WARMUP + i + 1]
        )

        # CPU staging copy
        pinned[nxt].copy_(cpu_next)

        with torch.cuda.stream(copy_stream):

            gpu_raw[nxt].copy_(
                pinned[nxt],
                non_blocking=True
            )

            copy_done[nxt].record(
                copy_stream
            )

torch.cuda.synchronize()

wall_end = time.perf_counter()

overlap_total_ms = (
    wall_end - wall_start
) * 1000.0

overlap_ms_per_frame = (
    overlap_total_ms / NUM_FRAMES
)

In [ ]:
seq = np.asarray(sequential_times)

seq_median = np.median(seq)

print("\n" + "=" * 90)
print("STAGE 10B — TRANSFER / INFERENCE OVERLAP")
print("=" * 90)

print(
    f"Sequential pinned pipeline median : "
    f"{seq_median:.4f} ms/frame"
)

print(
    f"Overlapped total time             : "
    f"{overlap_total_ms:.4f} ms"
)

print(
    f"Overlapped average                : "
    f"{overlap_ms_per_frame:.4f} ms/frame"
)

print(
    f"Pipeline speedup                  : "
    f"{seq_median / overlap_ms_per_frame:.3f}x"
)

print(
    f"Sequential theoretical FPS        : "
    f"{1000 / seq_median:.2f}"
)

print(
    f"Overlapped throughput             : "
    f"{1000 / overlap_ms_per_frame:.2f} FPS"
)

In [ ]:
pd.DataFrame({
    "sequential_ms": sequential_times
}).to_csv(
    "/kaggle/working/stage10b_overlap.csv",
    index=False
)

print(
    "\nSaved: "
    "/kaggle/working/stage10b_overlap.csv"
)

In [ ]:
import cv2
import torch
import subprocess
import shutil

print("=" * 80)
print("STAGE 10C — GPU VIDEO DECODE ENVIRONMENT")
print("=" * 80)

print("GPU:", torch.cuda.get_device_name(0))

print("\nFFmpeg:")
print(shutil.which("ffmpeg"))

if shutil.which("ffmpeg"):
    result = subprocess.run(
        ["ffmpeg", "-hide_banner", "-hwaccels"],
        capture_output=True,
        text=True
    )
    print(result.stdout)
    print(result.stderr)

print("\nNVIDIA utilities:")

for cmd in [
    "nvidia-smi",
    "ffmpeg",
    "nvcc"
]:
    print(f"{cmd:12s}: {shutil.which(cmd)}")

print("\nOpenCV CUDA:")
print("cv2 version:", cv2.__version__)

try:
    print(
        "CUDA devices:",
        cv2.cuda.getCudaEnabledDeviceCount()
    )
except Exception as e:
    print("OpenCV CUDA unavailable:", e)

In [ ]:
!ffmpeg -hide_banner -decoders 2>/dev/null | grep -Ei "cuda|cuvid|nvdec|h264|hevc" | head -50
!ffmpeg -hide_banner -hwaccels

In [ ]:
!ffprobe -v error \
-show_entries stream=codec_name,codec_long_name,width,height,r_frame_rate \
-of default=noprint_wrappers=1 \
"/kaggle/input/datasets/tismajain/boxing/GH088416.mp4"

In [ ]:
!ffmpeg -hide_banner \
-c:v h264_cuvid \
-i "/kaggle/input/datasets/tismajain/boxing/GH088416.mp4" \
-frames:v 100 \
-f null - 2>&1 | tail -30

In [ ]:
# CPU decode benchmark
!ffmpeg -benchmark -hide_banner \
-i "/kaggle/input/datasets/tismajain/boxing/GH088416.mp4" \
-frames:v 300 \
-f null - 2>&1 | tail -20

In [ ]:
# NVDEC benchmark
!ffmpeg -benchmark -hide_banner \
-c:v h264_cuvid \
-i "/kaggle/input/datasets/tismajain/boxing/GH088416.mp4" \
-frames:v 300 \
-f null - 2>&1 | tail -20

In [ ]:
import importlib.util

print(
    "PyNvVideoCodec installed:",
    importlib.util.find_spec("PyNvVideoCodec") is not None
)

In [ ]:
!pip install -q PyNvVideoCodec

In [ ]:
import PyNvVideoCodec as nvc
print("PyNvVideoCodec imported successfully")
print(nvc)

In [ ]:
print([
    name for name in dir(nvc)
    if "Decoder" in name or "decode" in name.lower()
])

In [ ]:
import inspect
import PyNvVideoCodec as nvc

for obj in [
    nvc.CreateDecoder,
    nvc.CreateSimpleDecoder,
    nvc.SimpleDecoder,
    nvc.DecodedFrame,
]:
    print("\n" + "=" * 80)
    print(obj)
    print("=" * 80)

    try:
        print("Signature:", inspect.signature(obj))
    except Exception as e:
        print("Signature unavailable:", e)

    try:
        print("\nDocstring:")
        print(inspect.getdoc(obj))
    except Exception as e:
        print("Docstring unavailable:", e)

print("DecodedFrame members:")
print([
    x for x in dir(nvc.DecodedFrame)
    if not x.startswith("_")
])

print("SimpleDecoder members:")
print([
    x for x in dir(nvc.SimpleDecoder)
    if not x.startswith("_")
])

In [ ]:
import PyNvVideoCodec as nvc

VIDEO_PATH = "/kaggle/input/datasets/tismajain/boxing/GH088416.mp4"

decoder = nvc.SimpleDecoder(
    VIDEO_PATH,
    gpu_id=0,
    use_device_memory=True,
    output_color_type=nvc.OutputColorType.NATIVE
)

print("Decoder created")
print(decoder)

In [ ]:
frames = decoder.get_batch_frames(1)

print("Type:", type(frames))
print("Length:", len(frames))

frame0 = frames[0]

print("\nDecodedFrame type:", type(frame0))
print("format:", frame0.format)
print("shape:", frame0.shape)
print("strides:", frame0.strides)
print("dtype:", frame0.dtype)
print("cuda:", frame0.cuda)
print("framesize:", frame0.framesize)

print("\nCUDA array interface:")
print(frame0.__cuda_array_interface__ if hasattr(frame0, "__cuda_array_interface__") else "none")

for i in range(2):
    try:
        ptr = frame0.GetPtrToPlane(i)
        print(f"Plane {i} ptr:", ptr)
    except Exception as e:
        print(f"Plane {i} error:", e)

In [ ]:
print("cuda():", frame0.cuda())
print("framesize():", frame0.framesize())

In [ ]:
!ffprobe -v error \
-select_streams v:0 \
-show_entries stream=color_range,color_space,color_transfer,color_primaries,pix_fmt \
-of default=noprint_wrappers=1 \
"/kaggle/input/datasets/tismajain/boxing/GH088416.mp4"

In [ ]:
%%writefile /kaggle/working/nv12_yolo_preprocess.cu

#include <cuda_runtime.h>
#include <stdint.h>
#include <math.h>

// matrix_type:
// 0 = BT.601 limited range
// 1 = BT.709 limited range

__device__ __forceinline__
void yuv_to_rgb(
    float Y,
    float U,
    float V,
    float &R,
    float &G,
    float &B,
    int matrix_type
) {
    // Limited / TV range
    float y = 1.16438356f * (Y - 16.0f);
    float u = U - 128.0f;
    float v = V - 128.0f;

    if (matrix_type == 1) {
        // BT.709
        R = y + 1.79274107f * v;
        G = y - 0.21324861f * u - 0.53290933f * v;
        B = y + 2.11240179f * u;
    } else {
        // BT.601
        R = y + 1.59602678f * v;
        G = y - 0.39176229f * u - 0.81296765f * v;
        B = y + 2.01723214f * u;
    }

    R = fminf(255.0f, fmaxf(0.0f, R));
    G = fminf(255.0f, fmaxf(0.0f, G));
    B = fminf(255.0f, fmaxf(0.0f, B));
}


// Bilinear sample from NV12 after converting neighbors to RGB.
__device__ __forceinline__
void sample_nv12_rgb(
    const uint8_t* y_plane,
    const uint8_t* uv_plane,
    int width,
    int height,
    float src_x,
    float src_y,
    int matrix_type,
    float &R,
    float &G,
    float &B
) {
    int x0 = (int)floorf(src_x);
    int y0 = (int)floorf(src_y);

    float wx = src_x - x0;
    float wy = src_y - y0;

    int x1 = x0 + 1;
    int y1 = y0 + 1;

    x0 = max(0, min(x0, width - 1));
    x1 = max(0, min(x1, width - 1));
    y0 = max(0, min(y0, height - 1));
    y1 = max(0, min(y1, height - 1));

    float r[4], g[4], b[4];

    int xs[4] = {x0, x1, x0, x1};
    int ys[4] = {y0, y0, y1, y1};

    #pragma unroll
    for (int i = 0; i < 4; ++i) {

        int x = xs[i];
        int y = ys[i];

        float Y = (float)y_plane[y * width + x];

        // NV12:
        // one interleaved UV pair for each 2x2 Y block.
        int uv_x = (x / 2) * 2;
        int uv_y = y / 2;

        int uv_idx = uv_y * width + uv_x;

        float U = (float)uv_plane[uv_idx + 0];
        float V = (float)uv_plane[uv_idx + 1];

        yuv_to_rgb(
            Y, U, V,
            r[i], g[i], b[i],
            matrix_type
        );
    }

    float r_top = r[0] + wx * (r[1] - r[0]);
    float r_bot = r[2] + wx * (r[3] - r[2]);

    float g_top = g[0] + wx * (g[1] - g[0]);
    float g_bot = g[2] + wx * (g[3] - g[2]);

    float b_top = b[0] + wx * (b[1] - b[0]);
    float b_bot = b[2] + wx * (b[3] - b[2]);

    R = r_top + wy * (r_bot - r_top);
    G = g_top + wy * (g_bot - g_top);
    B = b_top + wy * (b_bot - b_top);
}


__global__ void fused_nv12_yolo(
    const uint8_t* __restrict__ y_plane,
    const uint8_t* __restrict__ uv_plane,
    float* __restrict__ output,
    int matrix_type
) {
    const int src_w = 1920;
    const int src_h = 1080;

    const int out_w = 640;
    const int out_h = 384;

    const int resized_h = 360;
    const int pad_top = 12;

    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;

    if (x >= out_w || y >= out_h)
        return;

    int pixel = y * out_w + x;
    int plane = out_w * out_h;

    // YOLO padding
    if (y < pad_top || y >= pad_top + resized_h) {

        float p = 114.0f / 255.0f;

        output[0 * plane + pixel] = p;
        output[1 * plane + pixel] = p;
        output[2 * plane + pixel] = p;

        return;
    }

    int ry = y - pad_top;

    // Exact same half-pixel resize mapping as before.
    float scale_x = (float)src_w / (float)out_w;
    float scale_y = (float)src_h / (float)resized_h;

    float src_x =
        (x + 0.5f) * scale_x - 0.5f;

    float src_y =
        (ry + 0.5f) * scale_y - 0.5f;

    float R, G, B;

    sample_nv12_rgb(
        y_plane,
        uv_plane,
        src_w,
        src_h,
        src_x,
        src_y,
        matrix_type,
        R, G, B
    );

    output[0 * plane + pixel] = R / 255.0f;
    output[1 * plane + pixel] = G / 255.0f;
    output[2 * plane + pixel] = B / 255.0f;
}


extern "C"
void launch_nv12_yolo(
    const uint8_t* y_plane,
    const uint8_t* uv_plane,
    float* output,
    int matrix_type,
    cudaStream_t stream
) {
    dim3 block(16,16);
    dim3 grid(
        (640 + 15) / 16,
        (384 + 15) / 16
    );

    fused_nv12_yolo<<<grid, block, 0, stream>>>(
        y_plane,
        uv_plane,
        output,
        matrix_type
    );
}

In [ ]:
!nvcc -O3 -shared -Xcompiler -fPIC \
/kaggle/working/nv12_yolo_preprocess.cu \
-o /kaggle/working/libnv12_yolo.so

In [ ]:
import ctypes
import torch

nv12lib = ctypes.CDLL(
    "/kaggle/working/libnv12_yolo.so"
)

nv12lib.launch_nv12_yolo.argtypes = [
    ctypes.c_void_p,   # Y
    ctypes.c_void_p,   # UV
    ctypes.c_void_p,   # output
    ctypes.c_int,      # matrix
    ctypes.c_void_p    # stream
]

In [ ]:
y_ptr = frame0.GetPtrToPlane(0)
uv_ptr = frame0.GetPtrToPlane(1)

nv12_601 = torch.empty(
    (1, 3, 384, 640),
    dtype=torch.float32,
    device="cuda"
)

nv12_709 = torch.empty_like(nv12_601)

stream = torch.cuda.current_stream()

# BT.601
nv12lib.launch_nv12_yolo(
    ctypes.c_void_p(y_ptr),
    ctypes.c_void_p(uv_ptr),
    ctypes.c_void_p(nv12_601.data_ptr()),
    0,
    ctypes.c_void_p(stream.cuda_stream)
)

# BT.709
nv12lib.launch_nv12_yolo(
    ctypes.c_void_p(y_ptr),
    ctypes.c_void_p(uv_ptr),
    ctypes.c_void_p(nv12_709.data_ptr()),
    1,
    ctypes.c_void_p(stream.cuda_stream)
)

torch.cuda.synchronize()

In [ ]:
diff601 = torch.abs(
    nv12_601 - ultra
)

diff709 = torch.abs(
    nv12_709 - ultra
)

print("=" * 80)
print("NV12 COLOR MATRIX VALIDATION")
print("=" * 80)

print("\nBT.601")
print(
    "Mean abs diff:",
    float(diff601.mean())
)
print(
    "Max abs diff :",
    float(diff601.max())
)

print("\nBT.709")
print(
    "Mean abs diff:",
    float(diff709.mean())
)
print(
    "Max abs diff :",
    float(diff709.max())
)

In [ ]:
!ffprobe -v error \
-select_streams v:0 \
-show_entries stream=chroma_location \
-of default=noprint_wrappers=1 \
"/kaggle/input/datasets/tismajain/boxing/GH088416.mp4"

print("NVDEC PTS:", frame0.getPTS())
print("NVDEC timestamp:", frame0.timestamp)

print(decoder.get_stream_metadata())

In [ ]:
import inspect

for name in [
    "get_batch_frames_by_index",
    "seek_to_index",
    "get_batch_frames"
]:
    fn = getattr(decoder, name)

    print("\n", name)
    try:
        print("signature:", inspect.signature(fn))
    except Exception as e:
        print("signature unavailable:", e)

    print("doc:", inspect.getdoc(fn))


print(inspect.getdoc(frame0.nv12_to_rgb))

try:
    print(inspect.signature(frame0.nv12_to_rgb))
except Exception as e:
    print("signature unavailable:", e)

In [ ]:
frame0_idx = decoder.get_batch_frames_by_index([0])[0]

print("PTS:", frame0_idx.getPTS())
print("timestamp:", frame0_idx.timestamp)
print("format:", frame0_idx.format)
print("shape:", frame0_idx.shape)

rgb_ref = frame0_idx.nv12_to_rgb()

print("RGB ref type :", type(rgb_ref))
print("RGB ref shape:", getattr(rgb_ref, "shape", None))
print("RGB ref dtype:", getattr(rgb_ref, "dtype", None))
print("RGB ref cuda :", getattr(rgb_ref, "cuda", None))

In [ ]:
print("After nv12_to_rgb()")

print("format:", frame0_idx.format)
print("shape :", frame0_idx.shape)
print("dtype :", frame0_idx.dtype)
print("cuda():", frame0_idx.cuda())
print("framesize():", frame0_idx.framesize())

for i in range(3):
    try:
        print(f"Plane {i} ptr:", frame0_idx.GetPtrToPlane(i))
    except Exception as e:
        print(f"Plane {i}:", e)

In [ ]:
import PyNvVideoCodec as nvc

rgb_decoder = nvc.SimpleDecoder(
    VIDEO_PATH,
    gpu_id=0,
    use_device_memory=True,
    output_color_type=nvc.OutputColorType.RGBP
)

rgb_frame0 = rgb_decoder.get_batch_frames_by_index([0])[0]

print("format     :", rgb_frame0.format)
print("shape      :", rgb_frame0.shape)
print("strides    :", rgb_frame0.strides)
print("dtype      :", rgb_frame0.dtype)
print("framesize  :", rgb_frame0.framesize())
print("cuda views :", rgb_frame0.cuda())


views = rgb_frame0.cuda()

print("num views:", len(views))

for i, v in enumerate(views):
    print(f"\nView {i}")
    print("type :", type(v))
    print("shape:", getattr(v, "shape", None))
    print("cuda array interface:")
    print(
        getattr(
            v,
            "__cuda_array_interface__",
            None
        )
    )

In [ ]:
import cupy as cp

class CudaArrayProxy:
    def __init__(self, obj):
        cai = dict(obj.__cuda_array_interface__)

        # CuPy expects tuples, not lists
        cai["shape"] = tuple(cai["shape"])

        if cai.get("strides") is not None:
            cai["strides"] = tuple(cai["strides"])

        self.__cuda_array_interface__ = cai


rgb_views = rgb_frame0.cuda()

r_cp = cp.asarray(CudaArrayProxy(rgb_views[0]))
g_cp = cp.asarray(CudaArrayProxy(rgb_views[1]))
b_cp = cp.asarray(CudaArrayProxy(rgb_views[2]))

print("CuPy shapes:")
print(r_cp.shape, g_cp.shape, b_cp.shape)

print("dtypes:")
print(r_cp.dtype, g_cp.dtype, b_cp.dtype)

In [ ]:
import torch
import torch.nn.functional as F

r_t = torch.utils.dlpack.from_dlpack(r_cp)
g_t = torch.utils.dlpack.from_dlpack(g_cp)
b_t = torch.utils.dlpack.from_dlpack(b_cp)

rgb_full = torch.stack(
    [r_t, g_t, b_t],
    dim=0
).unsqueeze(0)

print("RGB full shape :", rgb_full.shape)
print("dtype          :", rgb_full.dtype)
print("device         :", rgb_full.device)
print("range          :", int(rgb_full.min()), int(rgb_full.max()))

In [ ]:
rgb_ref = rgb_full.float() / 255.0

rgb_ref = F.interpolate(
    rgb_ref,
    size=(360, 640),
    mode="bilinear",
    align_corners=False
)

rgb_ref = F.pad(
    rgb_ref,
    (0, 0, 12, 12),
    mode="constant",
    value=114.0 / 255.0
)

print("Reference shape:", rgb_ref.shape)

In [ ]:
nv12_frame0 = decoder.get_batch_frames_by_index([0])[0]

y_ptr = nv12_frame0.GetPtrToPlane(0)
uv_ptr = nv12_frame0.GetPtrToPlane(1)

nv12_custom = torch.empty(
    (1, 3, 384, 640),
    dtype=torch.float32,
    device="cuda"
)

stream = torch.cuda.current_stream()

nv12lib.launch_nv12_yolo(
    ctypes.c_void_p(y_ptr),
    ctypes.c_void_p(uv_ptr),
    ctypes.c_void_p(nv12_custom.data_ptr()),
    0,  # BT.601
    ctypes.c_void_p(stream.cuda_stream)
)

torch.cuda.synchronize()

diff = torch.abs(nv12_custom - rgb_ref)

print("=" * 80)
print("NV12 FUSED vs GPU RGBP REFERENCE")
print("=" * 80)

print("Mean abs diff :", float(diff.mean()))
print("Max abs diff  :", float(diff.max()))

for c, name in enumerate(["R", "G", "B"]):
    d = diff[0, c]

    print(
        f"{name}: mean={float(d.mean()):.6f} "
        f"max={float(d.max()):.6f}"
    )

In [ ]:
%%writefile /kaggle/working/nv12_yolo_preprocess.cu


#include <cuda_runtime.h>
#include <stdint.h>
#include <math.h>

// ============================================================
// Limited-range YUV -> RGB
//
// matrix_type:
// 0 = BT.601 limited range
// 1 = BT.709 limited range
// ============================================================

__device__ __forceinline__
void yuv_to_rgb(
    float Y,
    float U,
    float V,
    float &R,
    float &G,
    float &B,
    int matrix_type
) {
    float y = 1.16438356f * (Y - 16.0f);
    float u = U - 128.0f;
    float v = V - 128.0f;

    if (matrix_type == 1) {
        // BT.709 limited range
        R = y + 1.79274107f * v;
        G = y - 0.21324861f * u - 0.53290933f * v;
        B = y + 2.11240179f * u;
    } else {
        // BT.601 limited range
        R = y + 1.59602678f * v;
        G = y - 0.39176229f * u - 0.81296765f * v;
        B = y + 2.01723214f * u;
    }

    R = fminf(255.0f, fmaxf(0.0f, R));
    G = fminf(255.0f, fmaxf(0.0f, G));
    B = fminf(255.0f, fmaxf(0.0f, B));
}


// ============================================================
// Bilinear UV sampling for NV12
//
// Video chroma_location = left
//
// NV12 layout:
// Y plane  : H x W
// UV plane : H/2 x W bytes
//            interleaved U,V,U,V...
// ============================================================

__device__ __forceinline__
void sample_uv_bilinear(
    const uint8_t* uv_plane,
    int width,
    int height,
    float src_x,
    float src_y,
    float &U,
    float &V
) {
    const int uv_width  = width / 2;
    const int uv_height = height / 2;

    // Left chroma siting:
    // horizontal chroma sample k is aligned with x = 2*k
    //
    // Vertically, 4:2:0 chroma is centered between
    // corresponding luma rows.
    float cx = src_x * 0.5f;
    float cy = (src_y - 0.5f) * 0.5f;

    int x0 = (int)floorf(cx);
    int y0 = (int)floorf(cy);

    float wx = cx - x0;
    float wy = cy - y0;

    int x1 = x0 + 1;
    int y1 = y0 + 1;

    x0 = max(0, min(x0, uv_width - 1));
    x1 = max(0, min(x1, uv_width - 1));

    y0 = max(0, min(y0, uv_height - 1));
    y1 = max(0, min(y1, uv_height - 1));

    // Each chroma sample is two bytes: U,V
    int idx00 = y0 * width + 2 * x0;
    int idx01 = y0 * width + 2 * x1;
    int idx10 = y1 * width + 2 * x0;
    int idx11 = y1 * width + 2 * x1;

    float u00 = (float)uv_plane[idx00 + 0];
    float v00 = (float)uv_plane[idx00 + 1];

    float u01 = (float)uv_plane[idx01 + 0];
    float v01 = (float)uv_plane[idx01 + 1];

    float u10 = (float)uv_plane[idx10 + 0];
    float v10 = (float)uv_plane[idx10 + 1];

    float u11 = (float)uv_plane[idx11 + 0];
    float v11 = (float)uv_plane[idx11 + 1];

    float u_top = u00 + wx * (u01 - u00);
    float u_bot = u10 + wx * (u11 - u10);

    float v_top = v00 + wx * (v01 - v00);
    float v_bot = v10 + wx * (v11 - v10);

    U = u_top + wy * (u_bot - u_top);
    V = v_top + wy * (v_bot - v_top);
}


// ============================================================
// Bilinear luma + bilinear chroma + YUV -> RGB
// ============================================================

__device__ __forceinline__
void sample_nv12_rgb(
    const uint8_t* y_plane,
    const uint8_t* uv_plane,
    int width,
    int height,
    float src_x,
    float src_y,
    int matrix_type,
    float &R,
    float &G,
    float &B
) {
    // --------------------------------------------------------
    // Bilinear Y sampling
    // --------------------------------------------------------

    int x0 = (int)floorf(src_x);
    int y0 = (int)floorf(src_y);

    float wx = src_x - x0;
    float wy = src_y - y0;

    int x1 = x0 + 1;
    int y1 = y0 + 1;

    x0 = max(0, min(x0, width - 1));
    x1 = max(0, min(x1, width - 1));

    y0 = max(0, min(y0, height - 1));
    y1 = max(0, min(y1, height - 1));

    float y00 = (float)y_plane[y0 * width + x0];
    float y01 = (float)y_plane[y0 * width + x1];
    float y10 = (float)y_plane[y1 * width + x0];
    float y11 = (float)y_plane[y1 * width + x1];

    float y_top = y00 + wx * (y01 - y00);
    float y_bot = y10 + wx * (y11 - y10);

    float Y = y_top + wy * (y_bot - y_top);

    // --------------------------------------------------------
    // Bilinear UV sampling
    // --------------------------------------------------------

    float U, V;

    sample_uv_bilinear(
        uv_plane,
        width,
        height,
        src_x,
        src_y,
        U,
        V
    );

    // --------------------------------------------------------
    // Convert to RGB
    // --------------------------------------------------------

    yuv_to_rgb(
        Y,
        U,
        V,
        R,
        G,
        B,
        matrix_type
    );
}


// ============================================================
// Fused NV12 -> YOLO tensor
//
// Input:
//   NV12 GPU-resident Y and UV planes
//
// Output:
//   float32 RGB CHW
//   shape = [3, 384, 640]
//
// Operations fused:
//   NV12 -> RGB
//   bilinear resize
//   letterbox
//   CHW layout
//   normalization
// ============================================================

__global__
void fused_nv12_yolo(
    const uint8_t* __restrict__ y_plane,
    const uint8_t* __restrict__ uv_plane,
    float* __restrict__ output,
    int matrix_type
) {
    const int src_w = 1920;
    const int src_h = 1080;

    const int out_w = 640;
    const int out_h = 384;

    const int resized_w = 640;
    const int resized_h = 360;

    const int pad_left = 0;
    const int pad_top = 12;

    int x =
        blockIdx.x * blockDim.x +
        threadIdx.x;

    int y =
        blockIdx.y * blockDim.y +
        threadIdx.y;

    if (x >= out_w || y >= out_h)
        return;

    int pixel =
        y * out_w + x;

    int plane =
        out_w * out_h;

    // --------------------------------------------------------
    // Letterbox padding
    // --------------------------------------------------------

    if (
        x < pad_left ||
        x >= pad_left + resized_w ||
        y < pad_top ||
        y >= pad_top + resized_h
    ) {
        float pad =
            114.0f / 255.0f;

        output[
            0 * plane + pixel
        ] = pad;

        output[
            1 * plane + pixel
        ] = pad;

        output[
            2 * plane + pixel
        ] = pad;

        return;
    }

    // --------------------------------------------------------
    // Position inside resized image
    // --------------------------------------------------------

    int rx = x - pad_left;
    int ry = y - pad_top;

    // --------------------------------------------------------
    // Half-pixel resize mapping
    // --------------------------------------------------------

    float scale_x =
        (float)src_w /
        (float)resized_w;

    float scale_y =
        (float)src_h /
        (float)resized_h;

    float src_x =
        (rx + 0.5f) * scale_x -
        0.5f;

    float src_y =
        (ry + 0.5f) * scale_y -
        0.5f;

    // --------------------------------------------------------
    // NV12 sample -> RGB
    // --------------------------------------------------------

    float R, G, B;

    sample_nv12_rgb(
        y_plane,
        uv_plane,
        src_w,
        src_h,
        src_x,
        src_y,
        matrix_type,
        R,
        G,
        B
    );

    // --------------------------------------------------------
    // RGB -> normalized CHW
    // --------------------------------------------------------

    output[
        0 * plane + pixel
    ] = R / 255.0f;

    output[
        1 * plane + pixel
    ] = G / 255.0f;

    output[
        2 * plane + pixel
    ] = B / 255.0f;
}


// ============================================================
// Launcher
// ============================================================

extern "C"
void launch_nv12_yolo(
    const uint8_t* y_plane,
    const uint8_t* uv_plane,
    float* output,
    int matrix_type,
    cudaStream_t stream
) {
    dim3 block(
        16,
        16
    );

    dim3 grid(
        (640 + block.x - 1) /
            block.x,

        (384 + block.y - 1) /
            block.y
    );

    fused_nv12_yolo<<<
        grid,
        block,
        0,
        stream
    >>>(
        y_plane,
        uv_plane,
        output,
        matrix_type
    );
}

In [ ]:
!nvcc -O3 -shared -Xcompiler -fPIC \
/kaggle/working/nv12_yolo_preprocess.cu \
-o /kaggle/working/libnv12_yolo.so

In [ ]:
import ctypes

nv12lib = ctypes.CDLL(
    "/kaggle/working/libnv12_yolo.so"
)

nv12lib.launch_nv12_yolo.argtypes = [
    ctypes.c_void_p,
    ctypes.c_void_p,
    ctypes.c_void_p,
    ctypes.c_int,
    ctypes.c_void_p
]

In [ ]:
nv12_frame0 = decoder.get_batch_frames_by_index([0])[0]

y_ptr = nv12_frame0.GetPtrToPlane(0)
uv_ptr = nv12_frame0.GetPtrToPlane(1)

nv12_custom = torch.empty(
    (1, 3, 384, 640),
    dtype=torch.float32,
    device="cuda"
)

stream = torch.cuda.current_stream()

nv12lib.launch_nv12_yolo(
    ctypes.c_void_p(y_ptr),
    ctypes.c_void_p(uv_ptr),
    ctypes.c_void_p(
        nv12_custom.data_ptr()
    ),
    0,  # BT.601
    ctypes.c_void_p(
        stream.cuda_stream
    )
)

torch.cuda.synchronize()

In [ ]:
diff = torch.abs(
    nv12_custom - rgb_ref
)

print("=" * 80)
print("NV12 BILINEAR-CHROMA vs GPU RGBP")
print("=" * 80)

print(
    "Mean abs diff :",
    float(diff.mean())
)

print(
    "Max abs diff  :",
    float(diff.max())
)

for c, name in enumerate(
    ["R", "G", "B"]
):
    d = diff[0, c]

    print(
        f"{name}: "
        f"mean={float(d.mean()):.6f} "
        f"max={float(d.max()):.6f}"
    )

In [ ]:
import ctypes
import numpy as np
import torch
import torch.nn.functional as F

NUM_CHECK = 100
CONF = 0.25
IOU = 0.7

count_matches = 0
class_matches = 0
total_frames = 0

max_conf_diff = 0.0
max_box_diff_px = 0.0
mean_box_diffs = []

rgb_decoder = nvc.SimpleDecoder(
    VIDEO_PATH,
    gpu_id=0,
    use_device_memory=True,
    output_color_type=nvc.OutputColorType.RGBP
)

nv12_decoder = nvc.SimpleDecoder(
    VIDEO_PATH,
    gpu_id=0,
    use_device_memory=True,
    output_color_type=nvc.OutputColorType.NATIVE
)

class CudaArrayProxy:
    def __init__(self, obj):
        cai = dict(obj.__cuda_array_interface__)
        cai["shape"] = tuple(cai["shape"])
        if cai.get("strides") is not None:
            cai["strides"] = tuple(cai["strides"])
        self.__cuda_array_interface__ = cai


def rgbp_to_yolo_tensor(rgb_frame):
    views = rgb_frame.cuda()

    r_cp = cp.asarray(CudaArrayProxy(views[0]))
    g_cp = cp.asarray(CudaArrayProxy(views[1]))
    b_cp = cp.asarray(CudaArrayProxy(views[2]))

    r = torch.utils.dlpack.from_dlpack(r_cp)
    g = torch.utils.dlpack.from_dlpack(g_cp)
    b = torch.utils.dlpack.from_dlpack(b_cp)

    x = torch.stack([r, g, b], dim=0).unsqueeze(0)

    x = x.float() / 255.0

    x = F.interpolate(
        x,
        size=(360, 640),
        mode="bilinear",
        align_corners=False
    )

    x = F.pad(
        x,
        (0, 0, 12, 12),
        mode="constant",
        value=114.0 / 255.0
    )

    return x


def nv12_to_yolo_tensor(nv12_frame):
    y_ptr = nv12_frame.GetPtrToPlane(0)
    uv_ptr = nv12_frame.GetPtrToPlane(1)

    out = torch.empty(
        (1, 3, 384, 640),
        dtype=torch.float32,
        device="cuda"
    )

    stream = torch.cuda.current_stream()

    nv12lib.launch_nv12_yolo(
        ctypes.c_void_p(y_ptr),
        ctypes.c_void_p(uv_ptr),
        ctypes.c_void_p(out.data_ptr()),
        0,  # BT.601
        ctypes.c_void_p(stream.cuda_stream)
    )

    return out


for idx in range(NUM_CHECK):

    # -------------------------------------------------------
    # Decode SAME frame index through both GPU paths
    # -------------------------------------------------------

    rgb_frame = rgb_decoder.get_batch_frames_by_index([idx])[0]
    nv12_frame = nv12_decoder.get_batch_frames_by_index([idx])[0]

    rgb_x = rgbp_to_yolo_tensor(rgb_frame)
    nv12_x = nv12_to_yolo_tensor(nv12_frame)

    torch.cuda.synchronize()

    # -------------------------------------------------------
    # YOLO inference
    # -------------------------------------------------------

    res_rgb = model.predict(
        source=rgb_x,
        conf=CONF,
        iou=IOU,
        device=0,
        verbose=False
    )[0]

    res_nv12 = model.predict(
        source=nv12_x,
        conf=CONF,
        iou=IOU,
        device=0,
        verbose=False
    )[0]

    boxes_r = res_rgb.boxes
    boxes_n = res_nv12.boxes

    nr = 0 if boxes_r is None else len(boxes_r)
    nn = 0 if boxes_n is None else len(boxes_n)

    total_frames += 1

    if nr == nn:
        count_matches += 1

    if nr > 0 and nr == nn:

        conf_r = boxes_r.conf.detach().cpu()
        conf_n = boxes_n.conf.detach().cpu()

        cls_r = boxes_r.cls.detach().cpu()
        cls_n = boxes_n.cls.detach().cpu()

        xyxy_r = boxes_r.xyxy.detach().cpu()
        xyxy_n = boxes_n.xyxy.detach().cpu()

        # Sort by confidence for stable alignment
        order_r = torch.argsort(conf_r, descending=True)
        order_n = torch.argsort(conf_n, descending=True)

        conf_r = conf_r[order_r]
        conf_n = conf_n[order_n]

        cls_r = cls_r[order_r]
        cls_n = cls_n[order_n]

        xyxy_r = xyxy_r[order_r]
        xyxy_n = xyxy_n[order_n]

        if torch.equal(cls_r, cls_n):
            class_matches += 1

        conf_diff = torch.max(
            torch.abs(conf_r - conf_n)
        ).item()

        box_abs = torch.abs(
            xyxy_r - xyxy_n
        )

        max_conf_diff = max(
            max_conf_diff,
            conf_diff
        )

        max_box_diff_px = max(
            max_box_diff_px,
            torch.max(box_abs).item()
        )

        mean_box_diffs.append(
            torch.mean(box_abs).item()
        )

    if (idx + 1) % 20 == 0:
        print(
            f"Checked {idx + 1}/{NUM_CHECK}"
        )


print("\n" + "=" * 90)
print("STAGE 10D — GPU-RESIDENT YOLO EQUIVALENCE")
print("=" * 90)

print(
    f"Frames checked                  : "
    f"{total_frames}"
)

print(
    f"Matching detection counts       : "
    f"{count_matches}/{total_frames}"
)

print(
    f"Detection-count match rate      : "
    f"{100.0 * count_matches / total_frames:.2f}%"
)

print(
    f"Matching class sequences        : "
    f"{class_matches}/{total_frames}"
)

print(
    f"\nMax confidence difference        : "
    f"{max_conf_diff:.8f}"
)

print(
    f"Max box-coordinate difference   : "
    f"{max_box_diff_px:.6f} px"
)

if mean_box_diffs:
    print(
        f"Mean box-coordinate difference  : "
        f"{np.mean(mean_box_diffs):.6f} px"
    )

In [ ]:
%%writefile /kaggle/working/rgbp_yolo_preprocess.cu

#include <cuda_runtime.h>
#include <stdint.h>
#include <math.h>

__global__ void fused_rgbp_yolo(
    const uint8_t* __restrict__ r_plane,
    const uint8_t* __restrict__ g_plane,
    const uint8_t* __restrict__ b_plane,
    float* __restrict__ output,
    int src_w,
    int src_h
) {
    const int out_w = 640;
    const int out_h = 384;

    const int resized_w = 640;
    const int resized_h = 360;

    const int pad_top = 12;

    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;

    if (x >= out_w || y >= out_h)
        return;

    int out_idx = y * out_w + x;
    int plane = out_w * out_h;

    // Letterbox padding
    if (y < pad_top || y >= pad_top + resized_h) {

        float p = 114.0f / 255.0f;

        output[0 * plane + out_idx] = p;
        output[1 * plane + out_idx] = p;
        output[2 * plane + out_idx] = p;

        return;
    }

    int ry = y - pad_top;

    // Half-pixel resize mapping
    float scale_x = (float)src_w / (float)resized_w;
    float scale_y = (float)src_h / (float)resized_h;

    float src_x = (x  + 0.5f) * scale_x - 0.5f;
    float src_y = (ry + 0.5f) * scale_y - 0.5f;

    int x0 = (int)floorf(src_x);
    int y0 = (int)floorf(src_y);

    float wx = src_x - x0;
    float wy = src_y - y0;

    int x1 = x0 + 1;
    int y1 = y0 + 1;

    x0 = max(0, min(x0, src_w - 1));
    x1 = max(0, min(x1, src_w - 1));

    y0 = max(0, min(y0, src_h - 1));
    y1 = max(0, min(y1, src_h - 1));

    int idx00 = y0 * src_w + x0;
    int idx01 = y0 * src_w + x1;
    int idx10 = y1 * src_w + x0;
    int idx11 = y1 * src_w + x1;

    const uint8_t* planes[3] = {
        r_plane,
        g_plane,
        b_plane
    };

    #pragma unroll
    for (int c = 0; c < 3; ++c) {

        const uint8_t* src = planes[c];

        float v00 = (float)src[idx00];
        float v01 = (float)src[idx01];
        float v10 = (float)src[idx10];
        float v11 = (float)src[idx11];

        float top =
            v00 + wx * (v01 - v00);

        float bottom =
            v10 + wx * (v11 - v10);

        float value =
            top + wy * (bottom - top);

        output[c * plane + out_idx] =
            value / 255.0f;
    }
}


extern "C"
void launch_rgbp_yolo(
    const uint8_t* r_plane,
    const uint8_t* g_plane,
    const uint8_t* b_plane,
    float* output,
    int src_w,
    int src_h,
    cudaStream_t stream
) {
    dim3 block(16, 16);

    dim3 grid(
        (640 + 15) / 16,
        (384 + 15) / 16
    );

    fused_rgbp_yolo<<<
        grid,
        block,
        0,
        stream
    >>>(
        r_plane,
        g_plane,
        b_plane,
        output,
        src_w,
        src_h
    );
}

In [ ]:
!nvcc -O3 -shared -Xcompiler -fPIC \
/kaggle/working/rgbp_yolo_preprocess.cu \
-o /kaggle/working/librgbp_yolo.so

In [ ]:
import ctypes

rgbplib = ctypes.CDLL(
    "/kaggle/working/librgbp_yolo.so"
)

rgbplib.launch_rgbp_yolo.argtypes = [
    ctypes.c_void_p,
    ctypes.c_void_p,
    ctypes.c_void_p,
    ctypes.c_void_p,
    ctypes.c_int,
    ctypes.c_int,
    ctypes.c_void_p
]

In [ ]:
views = rgb_frame0.cuda()

r_ptr = views[0].__cuda_array_interface__["data"][0]
g_ptr = views[1].__cuda_array_interface__["data"][0]
b_ptr = views[2].__cuda_array_interface__["data"][0]

rgbp_custom = torch.empty(
    (1, 3, 384, 640),
    dtype=torch.float32,
    device="cuda"
)

stream = torch.cuda.current_stream()

rgbplib.launch_rgbp_yolo(
    ctypes.c_void_p(r_ptr),
    ctypes.c_void_p(g_ptr),
    ctypes.c_void_p(b_ptr),
    ctypes.c_void_p(rgbp_custom.data_ptr()),
    1920,
    1080,
    ctypes.c_void_p(stream.cuda_stream)
)

torch.cuda.synchronize()

In [ ]:
diff = torch.abs(
    rgbp_custom - rgb_ref
)

print("=" * 80)
print("RGBP FUSED CUDA CORRECTNESS")
print("=" * 80)

print(
    "Mean abs diff:",
    float(diff.mean())
)

print(
    "Max abs diff :",
    float(diff.max())
)

print(
    "Allclose 1e-5:",
    torch.allclose(
        rgbp_custom,
        rgb_ref,
        atol=1e-5,
        rtol=1e-5
    )
)

In [ ]:
import cupy as cp
import torch
import torch.nn.functional as F
import ctypes

class CudaArrayProxy:
    def __init__(self, obj):
        cai = dict(obj.__cuda_array_interface__)
        cai["shape"] = tuple(cai["shape"])
        if cai.get("strides") is not None:
            cai["strides"] = tuple(cai["strides"])
        self.__cuda_array_interface__ = cai


# ============================================================
# 1. Decode ONE fresh RGBP frame
# ============================================================

fresh_rgb = rgb_decoder.get_batch_frames_by_index([0])[0]

views = fresh_rgb.cuda()

print("format :", fresh_rgb.format)
print("shape  :", fresh_rgb.shape)

# Keep fresh_rgb alive until ALL comparisons are finished.


# ============================================================
# 2. Build reference from EXACT SAME GPU planes
# ============================================================

r_cp = cp.asarray(CudaArrayProxy(views[0]))
g_cp = cp.asarray(CudaArrayProxy(views[1]))
b_cp = cp.asarray(CudaArrayProxy(views[2]))

r_t = torch.utils.dlpack.from_dlpack(r_cp)
g_t = torch.utils.dlpack.from_dlpack(g_cp)
b_t = torch.utils.dlpack.from_dlpack(b_cp)

rgb_full = torch.stack(
    [r_t, g_t, b_t],
    dim=0
).unsqueeze(0)

# This creates independent float GPU storage,
# so decoder-buffer reuse cannot affect the reference.
reference = rgb_full.float() / 255.0

reference = F.interpolate(
    reference,
    size=(360, 640),
    mode="bilinear",
    align_corners=False
)

reference = F.pad(
    reference,
    (0, 0, 12, 12),
    mode="constant",
    value=114.0 / 255.0
)

# Force completion before doing anything else.
torch.cuda.synchronize()


# ============================================================
# 3. Get raw pointers from the SAME fresh frame
# ============================================================

r_ptr = views[0].__cuda_array_interface__["data"][0]
g_ptr = views[1].__cuda_array_interface__["data"][0]
b_ptr = views[2].__cuda_array_interface__["data"][0]


# ============================================================
# 4. Run fused CUDA kernel
# ============================================================

custom = torch.empty(
    (1, 3, 384, 640),
    dtype=torch.float32,
    device="cuda"
)

stream = torch.cuda.current_stream()

rgbplib.launch_rgbp_yolo(
    ctypes.c_void_p(r_ptr),
    ctypes.c_void_p(g_ptr),
    ctypes.c_void_p(b_ptr),
    ctypes.c_void_p(custom.data_ptr()),
    1920,
    1080,
    ctypes.c_void_p(stream.cuda_stream)
)

torch.cuda.synchronize()


# ============================================================
# 5. Compare
# ============================================================

diff = torch.abs(custom - reference)

print("\n" + "=" * 80)
print("FRESH RGBP REFERENCE vs FUSED CUDA")
print("=" * 80)

print("Reference shape :", tuple(reference.shape))
print("Custom shape    :", tuple(custom.shape))

print("Mean abs diff   :", float(diff.mean()))
print("Max abs diff    :", float(diff.max()))

for c, name in enumerate(["R", "G", "B"]):
    d = diff[0, c]

    print(
        f"{name}: "
        f"mean={float(d.mean()):.8f} "
        f"max={float(d.max()):.8f}"
    )


# Compare simple image statistics before resizing

for c, name in enumerate(["R", "G", "B"]):
    print(
        name,
        "min=", int(rgb_full[0, c].min()),
        "max=", int(rgb_full[0, c].max()),
        "mean=", float(rgb_full[0, c].float().mean())
    )

In [ ]:
import ctypes
import numpy as np
import cupy as cp
import torch
import torch.nn.functional as F

NUM_CHECK = 100
CONF = 0.25
IOU = 0.7

count_matches = 0
class_matches = 0
total_frames = 0

max_conf_diff = 0.0
max_box_diff_px = 0.0
mean_box_diffs = []

class CudaArrayProxy:
    def __init__(self, obj):
        cai = dict(obj.__cuda_array_interface__)
        cai["shape"] = tuple(cai["shape"])
        if cai.get("strides") is not None:
            cai["strides"] = tuple(cai["strides"])
        self.__cuda_array_interface__ = cai


def rgbp_reference_tensor(rgb_frame):
    views = rgb_frame.cuda()

    r_cp = cp.asarray(CudaArrayProxy(views[0]))
    g_cp = cp.asarray(CudaArrayProxy(views[1]))
    b_cp = cp.asarray(CudaArrayProxy(views[2]))

    r_t = torch.utils.dlpack.from_dlpack(r_cp)
    g_t = torch.utils.dlpack.from_dlpack(g_cp)
    b_t = torch.utils.dlpack.from_dlpack(b_cp)

    x = torch.stack([r_t, g_t, b_t], dim=0).unsqueeze(0)

    x = x.float() / 255.0

    x = F.interpolate(
        x,
        size=(360, 640),
        mode="bilinear",
        align_corners=False
    )

    x = F.pad(
        x,
        (0, 0, 12, 12),
        mode="constant",
        value=114.0 / 255.0
    )

    return x


def rgbp_fused_tensor(rgb_frame):
    views = rgb_frame.cuda()

    r_ptr = views[0].__cuda_array_interface__["data"][0]
    g_ptr = views[1].__cuda_array_interface__["data"][0]
    b_ptr = views[2].__cuda_array_interface__["data"][0]

    out = torch.empty(
        (1, 3, 384, 640),
        dtype=torch.float32,
        device="cuda"
    )

    stream = torch.cuda.current_stream()

    rgbplib.launch_rgbp_yolo(
        ctypes.c_void_p(r_ptr),
        ctypes.c_void_p(g_ptr),
        ctypes.c_void_p(b_ptr),
        ctypes.c_void_p(out.data_ptr()),
        1920,
        1080,
        ctypes.c_void_p(stream.cuda_stream)
    )

    return out


for idx in range(NUM_CHECK):

    # One fresh frame, shared by both paths.
    # Important: keep rgb_frame alive until both predictions finish.
    rgb_frame = rgb_decoder.get_batch_frames_by_index([idx])[0]

    ref_x = rgbp_reference_tensor(rgb_frame)
    fused_x = rgbp_fused_tensor(rgb_frame)

    torch.cuda.synchronize()

    res_ref = model.predict(
        source=ref_x,
        conf=CONF,
        iou=IOU,
        device=0,
        verbose=False
    )[0]

    res_fused = model.predict(
        source=fused_x,
        conf=CONF,
        iou=IOU,
        device=0,
        verbose=False
    )[0]

    boxes_r = res_ref.boxes
    boxes_f = res_fused.boxes

    nr = 0 if boxes_r is None else len(boxes_r)
    nf = 0 if boxes_f is None else len(boxes_f)

    total_frames += 1

    if nr == nf:
        count_matches += 1

    if nr > 0 and nr == nf:

        conf_r = boxes_r.conf.detach().cpu()
        conf_f = boxes_f.conf.detach().cpu()

        cls_r = boxes_r.cls.detach().cpu()
        cls_f = boxes_f.cls.detach().cpu()

        xyxy_r = boxes_r.xyxy.detach().cpu()
        xyxy_f = boxes_f.xyxy.detach().cpu()

        # Same detections should have virtually identical confidences,
        # so confidence sort is fine here.
        order_r = torch.argsort(conf_r, descending=True)
        order_f = torch.argsort(conf_f, descending=True)

        conf_r = conf_r[order_r]
        conf_f = conf_f[order_f]

        cls_r = cls_r[order_r]
        cls_f = cls_f[order_f]

        xyxy_r = xyxy_r[order_r]
        xyxy_f = xyxy_f[order_f]

        if torch.equal(cls_r, cls_f):
            class_matches += 1

        conf_diff = torch.max(
            torch.abs(conf_r - conf_f)
        ).item()

        box_abs = torch.abs(
            xyxy_r - xyxy_f
        )

        max_conf_diff = max(
            max_conf_diff,
            conf_diff
        )

        max_box_diff_px = max(
            max_box_diff_px,
            torch.max(box_abs).item()
        )

        mean_box_diffs.append(
            torch.mean(box_abs).item()
        )

    if (idx + 1) % 20 == 0:
        print(f"Checked {idx + 1}/{NUM_CHECK}")


print("\n" + "=" * 90)
print("STAGE 10E — RGBP GPU-RESIDENT YOLO EQUIVALENCE")
print("=" * 90)

print(f"Frames checked                  : {total_frames}")
print(f"Matching detection counts       : {count_matches}/{total_frames}")
print(f"Detection-count match rate      : {100.0 * count_matches / total_frames:.2f}%")
print(f"Matching class sequences        : {class_matches}/{total_frames}")

print(f"\nMax confidence difference        : {max_conf_diff:.8f}")
print(f"Max box-coordinate difference   : {max_box_diff_px:.6f} px")

if mean_box_diffs:
    print(
        f"Mean box-coordinate difference  : "
        f"{np.mean(mean_box_diffs):.6f} px"
    )

In [ ]:
import time
import numpy as np
import pandas as pd
import torch
import ctypes

NUM_FRAMES = 300
WARMUP = 30

# Fresh decoder for fair sequential access
rgb_bench_decoder = nvc.SimpleDecoder(
    VIDEO_PATH,
    gpu_id=0,
    use_device_memory=True,
    output_color_type=nvc.OutputColorType.RGBP
)

# ------------------------------------------------------------
# Warmup
# ------------------------------------------------------------

for i in range(WARMUP):
    frame = rgb_bench_decoder.get_batch_frames_by_index([i])[0]

    views = frame.cuda()

    r_ptr = views[0].__cuda_array_interface__["data"][0]
    g_ptr = views[1].__cuda_array_interface__["data"][0]
    b_ptr = views[2].__cuda_array_interface__["data"][0]

    x = torch.empty(
        (1, 3, 384, 640),
        dtype=torch.float32,
        device="cuda"
    )

    stream = torch.cuda.current_stream()

    rgbplib.launch_rgbp_yolo(
        ctypes.c_void_p(r_ptr),
        ctypes.c_void_p(g_ptr),
        ctypes.c_void_p(b_ptr),
        ctypes.c_void_p(x.data_ptr()),
        1920,
        1080,
        ctypes.c_void_p(stream.cuda_stream)
    )

    with torch.no_grad():
        _ = model.model(x)

torch.cuda.synchronize()

print("Warmup complete.")

In [ ]:
nvdec_total_times = []
nvdec_decode_times = []
nvdec_pre_times = []
nvdec_infer_times = []

for i in range(WARMUP, WARMUP + NUM_FRAMES):

    torch.cuda.synchronize()

    # --------------------------------------------------------
    # Decode
    # --------------------------------------------------------

    t0 = time.perf_counter()

    frame = rgb_bench_decoder.get_batch_frames_by_index([i])[0]

    torch.cuda.synchronize()

    t1 = time.perf_counter()

    # --------------------------------------------------------
    # Fused RGBP preprocessing
    # --------------------------------------------------------

    views = frame.cuda()

    r_ptr = views[0].__cuda_array_interface__["data"][0]
    g_ptr = views[1].__cuda_array_interface__["data"][0]
    b_ptr = views[2].__cuda_array_interface__["data"][0]

    x = torch.empty(
        (1, 3, 384, 640),
        dtype=torch.float32,
        device="cuda"
    )

    stream = torch.cuda.current_stream()

    rgbplib.launch_rgbp_yolo(
        ctypes.c_void_p(r_ptr),
        ctypes.c_void_p(g_ptr),
        ctypes.c_void_p(b_ptr),
        ctypes.c_void_p(x.data_ptr()),
        1920,
        1080,
        ctypes.c_void_p(stream.cuda_stream)
    )

    torch.cuda.synchronize()

    t2 = time.perf_counter()

    # --------------------------------------------------------
    # YOLO
    # --------------------------------------------------------

    with torch.no_grad():
        _ = model.model(x)

    torch.cuda.synchronize()

    t3 = time.perf_counter()

    nvdec_decode_times.append((t1 - t0) * 1000)
    nvdec_pre_times.append((t2 - t1) * 1000)
    nvdec_infer_times.append((t3 - t2) * 1000)
    nvdec_total_times.append((t3 - t0) * 1000)

    if (i - WARMUP + 1) % 50 == 0:
        print(f"Completed {i - WARMUP + 1}/{NUM_FRAMES}")

In [ ]:
def report(name, arr):
    arr = np.asarray(arr)

    print(
        f"{name:28s}"
        f"median={np.median(arr):8.4f} ms | "
        f"p95={np.percentile(arr,95):8.4f} ms | "
        f"mean={np.mean(arr):8.4f} ms"
    )


print("\n" + "=" * 95)
print("STAGE 10F — NVDEC RGBP END-TO-END")
print("=" * 95)

report("NVDEC decode", nvdec_decode_times)
report("Fused RGBP preprocess", nvdec_pre_times)
report("YOLO inference", nvdec_infer_times)
report("Total", nvdec_total_times)

total_med = np.median(nvdec_total_times)

print("\n" + "-" * 95)

print(f"Median total         : {total_med:.4f} ms")
print(f"Theoretical FPS      : {1000.0 / total_med:.2f}")

In [ ]:
pd.DataFrame({
    "decode_ms": nvdec_decode_times,
    "preprocess_ms": nvdec_pre_times,
    "inference_ms": nvdec_infer_times,
    "total_ms": nvdec_total_times,
}).to_csv(
    "/kaggle/working/stage10f_nvdec_rgbp.csv",
    index=False
)

print("\nSaved: /kaggle/working/stage10f_nvdec_rgbp.csv")

In [ ]:
import cv2
import time
import numpy as np
import torch

NUM_FRAMES = 300
WARMUP = 30

# ============================================================
# A. OpenCV + Ultralytics preprocessing + YOLO
# ============================================================

cap = cv2.VideoCapture(VIDEO_PATH)

baseline_decode = []
baseline_pre = []
baseline_infer = []
baseline_total = []

# Warmup
for _ in range(WARMUP):
    ok, f = cap.read()
    if not ok:
        break

    x = ultra_preprocess(f)

    with torch.no_grad():
        _ = model.model(x)

torch.cuda.synchronize()

print("Baseline warmup complete.")

for i in range(NUM_FRAMES):

    torch.cuda.synchronize()
    t0 = time.perf_counter()

    ok, f = cap.read()
    if not ok:
        break

    t1 = time.perf_counter()

    x = ultra_preprocess(f)

    torch.cuda.synchronize()
    t2 = time.perf_counter()

    with torch.no_grad():
        _ = model.model(x)

    torch.cuda.synchronize()
    t3 = time.perf_counter()

    baseline_decode.append((t1 - t0) * 1000)
    baseline_pre.append((t2 - t1) * 1000)
    baseline_infer.append((t3 - t2) * 1000)
    baseline_total.append((t3 - t0) * 1000)

cap.release()

In [ ]:
# ============================================================
# B. Sequential NVDEC + fused RGBP + YOLO
# ============================================================

nvdec_seq = nvc.SimpleDecoder(
    VIDEO_PATH,
    gpu_id=0,
    use_device_memory=True,
    output_color_type=nvc.OutputColorType.RGBP
)

nvdec_decode = []
nvdec_pre = []
nvdec_infer = []
nvdec_total = []

# Warmup sequentially
for _ in range(WARMUP):

    frames_out = nvdec_seq.get_batch_frames(1)

    if not frames_out:
        break

    frame = frames_out[0]
    views = frame.cuda()

    r_ptr = views[0].__cuda_array_interface__["data"][0]
    g_ptr = views[1].__cuda_array_interface__["data"][0]
    b_ptr = views[2].__cuda_array_interface__["data"][0]

    x = torch.empty(
        (1, 3, 384, 640),
        dtype=torch.float32,
        device="cuda"
    )

    stream = torch.cuda.current_stream()

    rgbplib.launch_rgbp_yolo(
        ctypes.c_void_p(r_ptr),
        ctypes.c_void_p(g_ptr),
        ctypes.c_void_p(b_ptr),
        ctypes.c_void_p(x.data_ptr()),
        1920,
        1080,
        ctypes.c_void_p(stream.cuda_stream)
    )

    with torch.no_grad():
        _ = model.model(x)

torch.cuda.synchronize()

print("NVDEC warmup complete.")

for i in range(NUM_FRAMES):

    torch.cuda.synchronize()
    t0 = time.perf_counter()

    frames_out = nvdec_seq.get_batch_frames(1)

    if not frames_out:
        break

    frame = frames_out[0]

    torch.cuda.synchronize()
    t1 = time.perf_counter()

    views = frame.cuda()

    r_ptr = views[0].__cuda_array_interface__["data"][0]
    g_ptr = views[1].__cuda_array_interface__["data"][0]
    b_ptr = views[2].__cuda_array_interface__["data"][0]

    x = torch.empty(
        (1, 3, 384, 640),
        dtype=torch.float32,
        device="cuda"
    )

    stream = torch.cuda.current_stream()

    rgbplib.launch_rgbp_yolo(
        ctypes.c_void_p(r_ptr),
        ctypes.c_void_p(g_ptr),
        ctypes.c_void_p(b_ptr),
        ctypes.c_void_p(x.data_ptr()),
        1920,
        1080,
        ctypes.c_void_p(stream.cuda_stream)
    )

    torch.cuda.synchronize()
    t2 = time.perf_counter()

    with torch.no_grad():
        _ = model.model(x)

    torch.cuda.synchronize()
    t3 = time.perf_counter()

    nvdec_decode.append((t1 - t0) * 1000)
    nvdec_pre.append((t2 - t1) * 1000)
    nvdec_infer.append((t3 - t2) * 1000)
    nvdec_total.append((t3 - t0) * 1000)

In [ ]:
def summary(name, arr):
    a = np.asarray(arr)

    print(
        f"{name:28s}"
        f"median={np.median(a):8.4f} ms | "
        f"p95={np.percentile(a,95):8.4f} ms | "
        f"mean={np.mean(a):8.4f} ms"
    )


print("\n" + "=" * 100)
print("FINAL STAGE 10 — APPLES-TO-APPLES PIPELINE COMPARISON")
print("=" * 100)

print("\nOPENCV / CPU PATH")
summary("Decode", baseline_decode)
summary("Preprocess", baseline_pre)
summary("Inference", baseline_infer)
summary("TOTAL", baseline_total)

print("\nNVDEC / GPU-RESIDENT PATH")
summary("Decode", nvdec_decode)
summary("Fused preprocess", nvdec_pre)
summary("Inference", nvdec_infer)
summary("TOTAL", nvdec_total)

base_med = np.median(baseline_total)
nvdec_med = np.median(nvdec_total)

print("\n" + "-" * 100)

print(f"Baseline median total : {base_med:.4f} ms")
print(f"NVDEC median total    : {nvdec_med:.4f} ms")

print(
    f"End-to-end speedup    : "
    f"{base_med / nvdec_med:.3f}x"
)

print(
    f"Baseline throughput   : "
    f"{1000 / base_med:.2f} FPS"
)

print(
    f"NVDEC throughput      : "
    f"{1000 / nvdec_med:.2f} FPS"
)

print(
    f"Latency reduction     : "
    f"{100 * (1 - nvdec_med/base_med):.2f}%"
)

In [ ]:
def summary(name, arr):
    a = np.asarray(arr)

    print(
        f"{name:28s}"
        f"median={np.median(a):8.4f} ms | "
        f"p95={np.percentile(a,95):8.4f} ms | "
        f"mean={np.mean(a):8.4f} ms"
    )


print("\n" + "=" * 100)
print("FINAL STAGE 10 — APPLES-TO-APPLES PIPELINE COMPARISON")
print("=" * 100)

print("\nOPENCV / CPU PATH")
summary("Decode", baseline_decode)
summary("Preprocess", baseline_pre)
summary("Inference", baseline_infer)
summary("TOTAL", baseline_total)

print("\nNVDEC / GPU-RESIDENT PATH")
summary("Decode", nvdec_decode)
summary("Fused preprocess", nvdec_pre)
summary("Inference", nvdec_infer)
summary("TOTAL", nvdec_total)

base_med = np.median(baseline_total)
nvdec_med = np.median(nvdec_total)

print("\n" + "-" * 100)

print(f"Baseline median total : {base_med:.4f} ms")
print(f"NVDEC median total    : {nvdec_med:.4f} ms")

print(
    f"End-to-end speedup    : "
    f"{base_med / nvdec_med:.3f}x"
)

print(
    f"Baseline throughput   : "
    f"{1000 / base_med:.2f} FPS"
)

print(
    f"NVDEC throughput      : "
    f"{1000 / nvdec_med:.2f} FPS"
)

print(
    f"Latency reduction     : "
    f"{100 * (1 - nvdec_med/base_med):.2f}%"
)

In [ ]:
pd.DataFrame({
    "baseline_decode_ms": baseline_decode,
    "baseline_pre_ms": baseline_pre,
    "baseline_infer_ms": baseline_infer,
    "baseline_total_ms": baseline_total,

    "nvdec_decode_ms": nvdec_decode,
    "nvdec_pre_ms": nvdec_pre,
    "nvdec_infer_ms": nvdec_infer,
    "nvdec_total_ms": nvdec_total,
}).to_csv(
    "/kaggle/working/stage10_final_pipeline.csv",
    index=False
)